# اجرای مدل Qwen و ساخت درگاه اتصال

این نوت‌بوک مدل `Qwen/Qwen2.5-1.5B-Instruct` را داخل Runtime کولب اجرا می‌کند، یک سرویس FastAPI می‌سازد و با ngrok یک آدرس عمومی برای اتصال بک‌اند ایجاد می‌کند.

قبل از اجرا، از منوی **Runtime > Change runtime type** یک GPU انتخاب کنید.

In [ ]:
!pip install -q -U "transformers>=4.43" accelerate fastapi uvicorn pyngrok nest-asyncio

In [ ]:
import os
import threading
import nest_asyncio
import torch
import uvicorn

from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer
from pyngrok import ngrok

# توکن ngrok را از https://dashboard.ngrok.com/get-started/your-authtoken بگیرید.
# برای امنیت، بهتر است آن را در Colab Secrets ذخیره کنید.
os.environ.setdefault("NGROK_AUTHTOKEN", "")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
tokenizer = None
model = None
generation_lock = threading.Lock()

print(f"Loading {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
)
model.eval()
print(f"Model loaded on {model.device}")

In [ ]:
class ChatRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=4000)
    system: str | None = Field(default=None, max_length=4000)
    max_new_tokens: int = Field(default=256, ge=1, le=1024)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

class ChatResponse(BaseModel):
    answer: str
    model: str

def generate_answer(request: ChatRequest) -> str:
    messages = []
    if request.system:
        messages.append({"role": "system", "content": request.system})
    messages.append({"role": "user", "content": request.question})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    generation_args = {
        **inputs,
        "max_new_tokens": request.max_new_tokens,
        "repetition_penalty": 1.1,
        "do_sample": request.temperature > 0,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if request.temperature > 0:
        generation_args["temperature"] = request.temperature

    with generation_lock, torch.inference_mode():
        output = model.generate(**generation_args)
    prompt_length = inputs["input_ids"].shape[1]
    return tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True).strip()

@asynccontextmanager
async def lifespan(_):
    yield

app = FastAPI(title="Local Qwen Gateway", version="1.0.0", lifespan=lifespan)

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME}

@app.post("/v1/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    try:
        answer = generate_answer(request)
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc
    return ChatResponse(answer=answer, model=MODEL_NAME)

# بررسی سریع مدل قبل از عمومی کردن سرویس
print(generate_answer(ChatRequest(question="سلام، خودت را معرفی کن.", max_new_tokens=40)))

In [ ]:
ngrok_auth_token = os.environ.get("NGROK_AUTHTOKEN", "").strip()
if not ngrok_auth_token:
    raise ValueError("ابتدا مقدار NGROK_AUTHTOKEN را در سلول قبل وارد کنید.")

ngrok.set_auth_token(ngrok_auth_token)
nest_asyncio.apply()

server_config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

tunnel = ngrok.connect(PORT, "http")
PUBLIC_AI_SERVICE_URL = tunnel.public_url
print(f"PUBLIC_AI_SERVICE_URL = {PUBLIC_AI_SERVICE_URL}")
print(f"Health check: {PUBLIC_AI_SERVICE_URL}/health")
print("این Runtime کولب باید تا پایان استفاده از API روشن بماند.")

In [ ]:
# تست درگاه عمومی
import requests

test_response = requests.post(
    f"{PUBLIC_AI_SERVICE_URL}/v1/chat",
    json={"question": "سلام امروز چه روزیه؟"},
    timeout=120,
)
print(test_response.status_code)
print(test_response.json())